# Laboratorium: Optymalizacja decyzji biznesowych. Regresja logistyczna i podstawy Backpropagation

W poprzednim etapie nauczyliśmy się przewidywać ciągłe wartości (koszty leczenia) za pomocą Wielowymiarowej Regresji Liniowej i algorytmu spadku gradientu. Ożywialiśmy matematyczne "miski", korzystaliśmy z wektorów i szukaliśmy dna z pomocą pochodnych. Dzisiaj wchodzimy na kolejny poziom. Wcielasz się w rolę Głównego Analityka Danych (Data Scientist) w międzynarodowej sieci hoteli.

## Twój cel na dziś
Dział obrotu (Revenue Management) ma ogromny problem z gośćmi, którzy rezerwują pokoje, a potem w ostatniej chwili anulują pobyt. Pokoje stoją puste, a hotel traci setki tysięcy dolarów. Otrzymujesz prawdziwy zbiór danych z portugalskich hoteli (*Hotel Booking Demand*). Twoim zadaniem jest zbudowanie od zera modelu opartego na sztucznej inteligencji, który z wyprzedzeniem określi **szansę (prawdopodobieństwo)**, z jaką dany klient anuluje rezerwację.

Otrzymujesz do dyspozycji 5 kluczowych informacji o każdym gościu:
1. **Lead Time:** Czas (w dniach) od zrobienia rezerwacji do zaplanowanego przyjazdu.
2. **ADR (Average Daily Rate):** Średnia cena za jedną noc.
3. **Total of Special Requests:** Liczba życzeń specjalnych (np. łóżeczko dla dziecka, wysokie piętro).
4. **Previous Cancellations:** Liczba rezerwacji, które dany klient anulował w przeszłości.
5. **Required Car Parking Spaces:** Czy klient zawnioskował o miejsce parkingowe (1 - tak, 0 - nie).

## Od Regresji Liniowej do Regresji Logistycznej (Sztuczny Neuron)
Poprzednio obliczaliśmy ciągłą kwotę rachunku: $\hat{y} = w_1x_1 + w_2x_2 + \dots + b$.
Teraz jednak nie chcemy kwoty dolarowej, ale **decyzji** oraz **prawdopodobieństwa** (czyli wartości ściśle z przedziału od 0 do 1), gdzie 1 oznacza z całą pewnością anulację, a 0 - bezpieczny przyjazd.

Aby to osiągnąć, bierzemy znany Ci już wzór ($z = w_1x_1 + \dots + w_5x_5 + b$) i "przepuszczamy" jego wynik przez specjalną nieliniową funkcję aktywacji – **Sigmoidę**. To właśnie ona stanowi o tym, że budujemy dziś w pełni działający, pojedynczy sztuczny neuron!

Oto równanie przewidywań naszego dzisiejszego modelu:
$$a = \sigma(z) = \frac{1}{1 + e^{-z}}$$

### Co tu jest DANE, a co SZUKANE?

**1. DANE (Nasza historia rezerwacji):**
* **$x$ (cechy rezerwacji):** Znamy konkretne wartości wspomnianych wyżej 5 cech dla tysięcy historycznych gości.
* **$y$ (prawdziwy status):** Znamy z historii fakt, jak postąpił klient: anulował ($y=1$) lub przyjechał ($y=0$). Będziemy z nim weryfikować poprawność przewidywań ($a$).

**2. SZUKANE (To, czego musi "nauczyć się" nasza sieć):**
* **$w$ (wagi / *weights*):** Pięć mnożników, których nie znamy. Zdefiniują one, jak silnie każda z cech wpływa na decyzję (np. czy brak miejsca parkingowego silniej zniechęca niż wysoka cena?).
* **$b$ (przesunięcie / *bias*):** Ogólna matematyczna tendencja do anulowania, stanowiąca punkt odniesienia dla reszty wag.

## Jak maszyna ma znaleźć te parametry? Pora na Backpropagation! (WYZWANIE)

Skoro zmienił się kształt i cel naszego modelu, Błąd Średniokwadratowy (MSE) z poprzednich zajęć przestał być odpowiedni. W problemach klasyfikacji binarnej stosujemy nową funkcję kosztu zwaną **Log-Loss** (lub *Binary Cross-Entropy*), opartą na logarytmach naturalnych:

$$J = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \ln(a_i) + (1 - y_i) \ln(1 - a_i) \right]$$

Twoim zadaniem, ramię w ramię z wiedzą z Analizy Matematycznej, będzie odnalezienie dna nowej doliny błędów. Aby znaleźć spadek gradientu dla tej skomplikowanej funkcji, zamiast liczyć ogromną pochodną "na raz", użyjemy **Reguły Łańcuchowej (Chain Rule)**.

Rozbijesz tę wielką pochodną na trzy mniejsze, bardzo proste do policzenia na kartce kroki (na tym właśnie polega słynna **Propagacja Wsteczna / Backpropagation**):
$$\frac{\partial J}{\partial w_1} = \frac{\partial J}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_1}$$

Gdy poprawnie zróżniczkujesz te 3 mniejsze równania i wymnożysz je przez siebie, ku Twojemu zaskoczeniu skomplikowane ułamki pięknie się skrócą, a Ty otrzymasz elegancki i potężny wzór do aktualizacji swoich wag.

Gotowy? Zaczynamy!

In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# ZADANIE 1: ZINDYWIDUALIZUJ SWOJE DANE
# =====================================================================

# TODO: Zmień poniższą wartość na swój faktyczny numer indeksu!
NUMER_INDEKSU = 258309

# Wczytujemy oryginalny zbiór danych (bezpośrednio z publicznego repozytorium GitHub)
# Zbiór "Hotel Booking Demand" zawiera ponad 100 tysięcy rezerwacji.
url = 'https://raw.githubusercontent.com/rkasa01/DATA607_Project2_Data/main/hotel_bookings.csv'
df_full = pd.read_csv(url)

# Używamy Twojego indeksu jako ziarna losowości (seed),
# aby wylosować Twój unikalny podzbiór 300 rezerwacji.
# Dzięki temu każdy w grupie będzie miał nieco inne wyniki końcowe!
df_student = df_full.sample(n=300, random_state=NUMER_INDEKSU).copy()


# =====================================================================
# PRZYGOTOWANIE DANYCH (PREPROCESSING)
# =====================================================================

# Definiujemy nasze 5 kluczowych cech, na podstawie których model podejmie decyzję
cechy = [
    'lead_time',
    'adr',
    'total_of_special_requests',
    'previous_cancellations',
    'required_car_parking_spaces'
]

# Wyciągamy interesujące nas kolumny z tabeli i zamieniamy je na macierze numpy.
# X_data to nasza macierz wejściowa o rozmiarze (300 rezerwacji, 5 cech)
X_data = df_student[cechy].values

# y_data to nasz wektor z prawdziwym statusem (1 = anulowana, 0 = zrealizowana)
y_data = df_student['is_canceled'].values

# Wyświetlamy podsumowanie operacji, aby upewnić się, że wszystko działa
print(f"✅ Wygenerowano unikalny zbiór dla indeksu: {NUMER_INDEKSU}")
print(f"Kształt macierzy danych X: {X_data.shape} (liczba rezerwacji, liczba cech)")
print(f"Kształt wektora decyzji y: {y_data.shape}")
print("\nOto pierwsze 5 wierszy z Twoich unikalnych danych biznesowych:")
print("-" * 90)
print(df_student[cechy + ['is_canceled']].head())
print("-" * 90)

## Krok 1: Pułapka skali, czyli dlaczego potrzebujemy Standaryzacji?

Zanim rzucimy nasz algorytm do walki, przyjrzyj się wartościom w wygenerowanej przed chwilą tabeli. Zmienna `lead_time` może wynosić np. 250 dni, podczas gdy `required_car_parking_spaces` przyjmuje wartości tylko 0 lub 1.

W poprzednim laboratorium, funkcja kosztu MSE przypominała ładną, okrągłą miskę. Przy tak ogromnych różnicach w skali naszych danych, nasza nowa funkcja kosztu Log-Loss będzie przypominać **bardzo wąski i niesamowicie długi kanion**. Algorytm spadku gradientu będzie odbijał się od jego ścian i może nigdy nie dotrzeć na dno (do minimum globalnego).

**Twoje pierwsze zadanie w kodzie:**
Zaimplementuj standaryzację (Z-score normalization) dla naszej macierzy $X$. Chcemy, aby każda z 5 kolumn miała średnią równą 0 i odchylenie standardowe równe 1. Wykorzystaj znany z analizy matematycznej wzór (gdzie $\mu$ to średnia, a $\sigma$ to odchylenie standardowe):

$$x_{norm} = \frac{x - \mu}{\sigma}$$

### Pro-Tip Analityka: Dlaczego używamy gotowych funkcji z biblioteki NumPy?

Patrząc na wzór na standaryzację, z pewnością potrafiłbyś napisać prostą pętlę `for` w Pythonie, która sumuje wszystkie wartości w kolumnie i dzieli je przez liczbę elementów, aby uzyskać $\mu$ (średnią), a następnie wylicza odchylenie standardowe. Dlaczego więc można użyć `np.mean()` oraz `np.std()`?

* **Wektoryzacja (Prędkość):** Pętle `for` w Pythonie są bardzo wolne. Biblioteka *NumPy* pod spodem jest napisana w wysoce zoptymalizowanym języku C. Zamiast przetwarzać dane wiersz po wierszu, potrafi wykonać te obliczenia na całych blokach pamięci jednocześnie (jest to tzw. wektoryzacja). W świecie AI, gdzie często przetwarzamy miliony rekordów, ta różnica decyduje o tym, czy Twój model uczy się 5 sekund, czy 5 godzin.
* **Niezawodność numeryczna:** Gotowe funkcje biblioteczne są zabezpieczone przed nietypowymi błędami (np. dzieleniem przez zero, gdyby jakaś kolumna miała same identyczne wartości i jej odchylenie wynosiło zero).
* **Skupienie na architekturze:** Jako inżynier AI powinieneś skupiać się na projektowaniu logiki uczenia i spadku gradientu, a nie wymyślać koła na nowo dla podstawowych operacji statystycznych. Używanie dedykowanych bibliotek matematycznych to absolutny standard branżowy.

---



In [ ]:
#TODO

Dobry Data Scientist potrafi nie tylko liczyć, ale też **pokazać** zależności w danych. Chociaż nasz zbiór ma wiele wymiarów (wiele cech), możemy wyciągnąć dwie z nich i zestawić je na płaskim wykresie.

Zestawmy ze sobą dwie bardzo ważne zmienne:
1. **Lead Time** (oś X) – ile dni wcześniej dokonano rezerwacji.
2. **Życzenia Specjalne** (oś Y) – o ile dodatkowych rzeczy prosił klient (np. łóżeczko, widok na morze).

Kolorami oznaczymy to, co faktycznie się wydarzyło:
*  **Zielony:** Gość przyjechał (Sukces).
*  **Czerwony:** Gość anulował (Problem).

###  Zadanie dla Ciebie:
Uruchom poniższy kod i przyjrzyj się układowi kropek na wykresie. Zastanów się:
* Czy czerwone i zielone punkty tworzą jakieś widoczne "skupiska"?
* Co można powiedzieć o klientach, którzy rezerwują pokój z rocznym wyprzedzeniem i nie mają żadnych życzeń?
* Gdzie na wykresie nasz model sztucznej inteligencji prawdopodobnie wyrysowałby swoją "linię odcięcia" (granicę decyzyjną)?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# PROSTA WIZUALIZACJA: Gdzie leży granica anulacji?
# =====================================================================

# Przyjmujemy poprawną nazwę danych: X_data
# Zakładamy kolejność kolumn: [lead_time, ..., total_of_special_requests, ...]
# Wydobywamy kolumny przy użyciu indeksów NumPy
lead_time = X_data[:, 0]
special_requests = X_data[:, 2]

# Rozdzielamy dane na dwie grupy na podstawie y_data
# (Tworzymy maski logiczne)
maska_przyjechali = (y_data == 0)
maska_anulowali = (y_data == 1)

# Tworzymy wykres
plt.figure(figsize=(10, 6))

# Rysujemy grupę "Przyjechali" (zielone kropki)
plt.scatter(lead_time[maska_przyjechali],
            special_requests[maska_przyjechali],
            color='forestgreen', alpha=0.5, label='Przyjechali (y=0)')

# Rysujemy grupę "Anulowali" (czerwone kropki)
plt.scatter(lead_time[maska_anulowali],
            special_requests[maska_anulowali],
            color='crimson', alpha=0.5, label='Anulowali (y=1)')

# Opisy osi i tytuł
plt.title('Zależność anulacji od czasu wyprzedzenia i życzeń specjalnych')
plt.xlabel('Lead Time (dni do przyjazdu)')
plt.ylabel('Liczba życzeń specjalnych')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

## Krok 2: Forward Pass (Krok w przód)

Nasz pojedynczy sztuczny neuron (Regresja Logistyczna) to proces dwuetapowy. Dla każdego pacjenta... przepraszam, dla każdego gościa hotelowego (przyzwyczajenia z poprzednich zajęć!), nasz model musi policzyć:

1. **Wynik liniowy:** $$z = w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4 + w_5x_5 + b$$
2. **Aktywację (Prawdopodobieństwo):** Przepuszczamy wynik przez funkcję sigmoidalną, która "ściśnie" dowolną liczbę do przedziału $(0, 1)$:
   $$a = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Wynikiem $a$ jest nasza diagnoza – procentowa szansa na to, że klient anuluje rezerwację.

---

## Krok 3: Fundamenty matematyczne – Regresja Logistyczna w notacji Jakobianowej

Zanim przejdziemy do implementacji, musimy precyzyjnie zdefiniować "przestrzeń operacyjną" naszego modelu. W uczeniu maszynowym precyzja oznaczeń jest kluczowa, aby uniknąć błędów w wymiarach macierzy (tzw. *dimension mismatch*).

### 1. Definicja zmiennych (DANE i PARAMETRY)

Pracujemy na pojedynczej obserwacji (jednej rezerwacji hotelowej), gdzie:

* **$\mathbf{x} \in \mathbb{R}^5$**: Wektor cech wejściowych (po standaryzacji). Jest to wektor kolumnowy: $\mathbf{x} = [x_1, x_2, x_3, x_4, x_5]^T$.
* **$y \in \{0, 1\}$**: Prawdziwa etykieta (skalar). Wartość $1$ oznacza anulowanie rezerwacji, $0$ oznacza jej realizację.
* **$\mathbf{w} \in \mathbb{R}^5$**: Wektor wag modelu. Szukamy optymalnych wartości $\mathbf{w} = [w_1, w_2, w_3, w_4, w_5]^T$, które określą wpływ każdej cechy na decyzję.
* **$b \in \mathbb{R}$**: Parametr przesunięcia (*bias*), będący liczbą rzeczywistą, określający "punkt startowy" predykcji przed uwzględnieniem cech.

### 2. Forward Pass – Model predykcyjny

Nasz model to funkcja złożona $f(\mathbf{x}) = \sigma(z(\mathbf{x}))$. Proces predykcji przebiega w dwóch etapach:

1.  **Funkcja liniowa $z$**:
    Obliczamy sumę ważoną cech powiększoną o bias:
    $$z(\mathbf{w}, b) = \mathbf{w}^T\mathbf{x} + b = \left( \sum_{j=1}^{5} w_j x_j \right) + b$$
    Wynik $z$ jest skalarem ($z \in \mathbb{R}$).

2.  **Funkcja aktywacji $a$ (Sigmoida)**:
    Aby zamienić dowolną wartość $z$ na prawdopodobieństwo, stosujemy funkcję logistyczną $\sigma$:
    $$a = \sigma(z) = \frac{1}{1 + e^{-z}}$$
    Wartość $a \in (0, 1)$ to nasza predykcja (prawdopodobieństwo anulowania). Ważna własność pochodnej sigmoidy, którą wykorzystasz, to: $\frac{da}{dz} = a(1-a)$.

### 3. Funkcja Straty (Binary Cross-Entropy)

Dla pojedynczej obserwacji definiujemy funkcję straty (Log-Loss), którą oznaczymy jako $L$, aby odróżnić ją od Jakobianu. Celem tej funkcji jest maksymalna penalizacja w sytuacji, gdy predykcja modelu $a \in (0, 1)$ diametralnie odbiega od prawdziwej etykiety $y \in \{0, 1\}$:
$$L(a, y) = - \left[ y \ln(a) + (1 - y) \ln(1 - a) \right]$$
Tutaj $L(a,y)$ traktujemy jako funkcję jednej zmiennej $a$,

### 4. Backpropagation – Rachunek Jakobianów (Reguła Łańcuchowa)

Aby zaktualizować wagi za pomocą algorytmu spadku gradientu, potrzebujemy wyznaczyć gradient funkcji straty względem wektora wag $\mathbf{w}$. W analizie wielowymiarowej zaczynamy od wyznaczenia **Jakobianu**, czyli wektora wierszowego pochodnych cząstkowych: $\mathbf{J}_{\mathbf{w}}(L) = \frac{\partial L}{\partial \mathbf{w}}$.

Zgodnie z twierdzeniem o pochodnej funkcji złożonej (reguła łańcuchowa), Jakobian ten jest iloczynem kolejnych Jakobianów poszczególnych warstw modelu:

$$\mathbf{J}_{\mathbf{w}}(L) = \underbrace{\mathbf{J}_{a}(L)}_{\text{skalar } 1\times 1} \cdot \underbrace{\mathbf{J}_{z}(a)}_{\text{skalar } 1\times 1} \cdot \underbrace{\mathbf{J}_{\mathbf{w}}(z)}_{\text{wektor wierszowy } 1\times 5}$$






## Zadanie 3: Twój ruch – Obliczenia na kartce

W poprzedniej komórce wprowadziliśmy potężne narzędzie: regułę łańcuchową dla Jakobianów. Czas ubrudzić sobie ręce matematyką!

Twoim zadaniem jest wyliczenie na kartce trzech składowych Jakobianów, a następnie wymnożenie ich przez siebie zgodnie ze wzorem:

$$\mathbf{J}_{\mathbf{w}}(L) = \mathbf{J}_{a}(L) \cdot \mathbf{J}_{z}(a) \cdot \mathbf{J}_{\mathbf{w}}(z)$$

Gdy to zrobisz, zapisz swój ostateczny (po wszystkich możliwych skrótach i uproszczeniach) wzór na **gradient** $\nabla_{\mathbf{w}} L$ w miejscu wskazanym poniżej. Pamiętaj, że ostateczny gradient, którego użyjemy w kodzie do aktualizacji wag, to wektor kolumnowy, czyli transpozycja Twojego wyliczonego Jakobianu: $\nabla_{\mathbf{w}} L = (\mathbf{J}_{\mathbf{w}}(L))^T$.

> ** PODPOWIEDŹ GŁÓWNEGO ANALITYKA:** > Jeśli Twój ostateczny wynik wygląda jak potwór z koszmarów (pełno w nim piętrowych ułamków, logarytmów i potęg), to znaczy, że gdzieś w obliczeniach wkradł się błąd. Magia matematyki w Regresji Logistycznej polega na tym, że skomplikowane mianowniki z pochodnej błędu (Log-Loss) **idealnie skracają się** z członami z pochodnej funkcji aktywacji (sigmoidy). Zaufaj matematyce – ostateczny wzór na gradient jest wręcz boleśnie prosty, elegancki i nie zawiera żadnych ułamków!

---

** MIEJSCE NA TWOJĄ ODPOWIEDŹ (Kliknij dwukrotnie tę komórkę, aby ją edytować) **

Po wymnożeniu składowych i skróceniu ułamków, ostateczny wzór na gradient funkcji straty względem wektora wag przyjmuje postać:

$$\nabla_{\mathbf{w}} L = \text{... (tutaj wpisz swój ostateczny wynik) ...}$$

## Krok 3.5: Gdzie podział się Bias? – Jakobian względem $b$

Wyliczyliśmy już, jak zmieniają się wagi $\mathbf{w}$, ale nasz model posiada jeszcze jeden kluczowy parametr: **bias $b$** (przesunięcie). Bez niego nasza funkcja aktywacji zawsze przechodziłaby przez punkt $(0,0)$ w przestrzeni cech, co drastycznie ograniczyłoby zdolność neuronu do dopasowania się do danych.

Z punktu widzenia analizy matematycznej, $b$ jest **skalarem**, więc jego Jakobian będzie po prostu zwykłą pochodną cząstkową (skalarem $1 \times 1$).

### Reguła łańcuchowa dla biasu

Stosujemy dokładnie tę samą ścieżkę, co w przypadku wag, ale na ostatnim etapie różniczkujemy funkcję liniową $z$ po zmiennej $b$:

$$\frac{\partial L}{\partial b} = \underbrace{\frac{\partial L}{\partial a}}_{\text{skalar}} \cdot \underbrace{\frac{\partial a}{\partial z}}_{\text{skalar}} \cdot \underbrace{\frac{\partial z}{\partial b}}_{\text{skalar}}$$

**Składowe dla biasu:**

1.  **$\frac{\partial L}{\partial a}$ oraz $\frac{\partial a}{\partial z}$**: Są identyczne jak w przypadku wag (już je znasz!).
2.  **Pochodna wyniku liniowego względem biasu ($\frac{\partial z}{\partial b}$):**
    Spójrzmy na definicję $z$:
    $$z = w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4 + w_5x_5 + b$$
    Różniczkując po $b$, wszystkie człony zawierające wagi i cechy ($w_ix_i$) stają się zerami, ponieważ nie zależą od $b$. Zostaje tylko:
    $$\frac{\partial z}{\partial b} = \frac{\partial}{\partial b}(b) = 1$$

### Ostateczny wynik dla $b$

Mnożąc wszystko przez siebie, otrzymujemy niesamowicie prostą zależność:
$$\frac{\partial L}{\partial b} =  \text{tutaj wpisz wynik}$$


## Krok 4: Optymalizacja modelu – Algorytm spadku gradientu (Gradient Descent)

Skoro wiemy już, jak policzyć błąd pojedynczej predykcji oraz jak ten błąd zmienia się pod wpływem zmian wag (czyli znamy gradient), możemy przejść do serca sztucznej inteligencji: **procesu uczenia**.

### 1. Funkcja Celu: Koszt Całkowity ($J$)

W uczeniu maszynowym nie optymalizujemy parametrów dla każdego gościa hotelowego z osobna. Naszym celem jest zbudowanie modelu, który będzie skuteczny globalnie. Dlatego algorytm spadku gradientu będziemy stosowali do **funkcji kosztu całkowitego** ($J$), która jest średnią arytmetyczną strat ($L$) poniesionych na wszystkich $m$ przykładach w zbiorze danych:

$$J(\mathbf{w}, b) = \frac{1}{m} \sum_{i=1}^{m} L(a^{(i)}, y^{(i)})$$

Matematycznie szukamy takich wartości parametrów $(\mathbf{w}, b)$, dla których funkcja $J$ przyjmuje swoje **minimum globalne**.

### 2. Mechanizm "schodzenia do doliny"

Wyobraź sobie, że funkcja kosztu $J$ to ukształtowanie terenu w wielowymiarowej przestrzeni, a Ty stoisz na zboczu góry w gęstej mgle. Nie widzisz, gdzie jest dno doliny, ale pod stopami czujesz nachylenie terenu.

Wyliczony przez Ciebie wcześniej **gradient** $\nabla J$ wskazuje kierunek najszybszego **wzrostu** funkcji. Aby zminimalizować błąd, musimy udać się w stronę dokładnie przeciwną.



### 3. Reguła aktualizacji (Update Rule)

W każdej iteracji algorytmu (tzw. epoce), będziemy aktualizować nasze wagi i bias według następujących wzorów:

1.  **Aktualizacja wag:**
    $$\mathbf{w} := \mathbf{w} - \alpha \cdot \nabla_{\mathbf{w}} J$$

2.  **Aktualizacja przesunięcia (bias):**
    $$b := b - \alpha \cdot \frac{\partial J}{\partial b}$$

**Gdzie kluczową rolę odgrywają:**
* **$\alpha$ (Learning Rate):** Współczynnik uczenia. To od niego zależy, jak duże kroki stawiamy. Jeśli będzie zbyt duży – możemy "przeskoczyć" minimum i nigdy go nie znaleźć. Jeśli zbyt mały – uczenie będzie trwało godzinami.
* **$\nabla J$:** Wektor gradientu, który dzięki Twoim obliczeniom na kartce przyjmie w kodzie bardzo prostą postać.

---
**Gotowy na kod?** W kolejnym kroku zainicjujemy wagi zerami i sprawdzimy, czy Twoje matematyczne wyprowadzenia pozwolą maszynie nauczyć się rozpoznawać rezygnacje gości!

In [ ]:
# =====================================================================
# ZADANIE 4: IMPLEMENTACJA SPADKU GRADIENTU (Pętla ucząca)
# =====================================================================


## Krok 5: TEST SYSTEMU – Ekstrakcja cech i predykcja

Twój model został wytrenowany. Czas sprawdzić go w boju! Poniżej znajduje się lista 7 nowych, fikcyjnych rezerwacji, które wpłynęły do systemu w postaci notatek z recepcji.

Twoim pierwszym zadaniem jako Analityka Danych jest **przełożenie tych tekstowych opisów na wektory liczbowe**. Pamiętaj, że w prawdziwym świecie dane rzadko przychodzą w idealnych tabelkach. Czeka na Ciebie sporo szumu informacyjnego – musisz wyciągnąć tylko te 5 cech, których używa nasz model: `Lead Time`, `Średnia Cena`, `Życzenia Specjalne` (ilość), `Długość Pobytu`, `Parking` (0 lub 1). Wszystkie inne informacje po prostu zignoruj.

Następnie, dla każdego pacj... to znaczy, gościa (jesteśmy w hotelu!), oblicz prawdopodobieństwo anulowania rezerwacji ($a$).

### Notatki z Recepcji:

**A. "Planista"**
> Pan Janusz rezerwuje pobyt z dokładnie rocznym wyprzedzeniem (365 dni). Zapłaci średnio 120 EUR za noc i zostaje na 5 nocy. Nie potrzebuje parkingu, bo przylatuje samolotem (ma lęk wysokości, więc prosił by o tym pamiętać). Nie ma absolutnie żadnych specjalnych życzeń co do pokoju. Wspomniał w mailu, że ma alergię na truskawki i zabiera ze sobą żonę, z którą świętuje 20. rocznicę ślubu.

**B. "Biznesmen"**
> Mark, dyrektor z londyńskiego banku. Rezerwuje apartament na zaledwie 2 dni przed przyjazdem. Płaci aż 250 EUR za nocleg. Zostaje tylko na 1 noc, bo jutro rano ma spotkanie zarządu. Wymaga miejsca parkingowego dla swojego wynajętego, czerwonego Porsche. Ma jedno, bardzo rygorystyczne życzenie: żąda podwójnego espresso do pokoju punktualnie o 6:00 rano. Nosi krawaty tylko w kolorze bordowym.

**C. "Rodzina 2+2"**
> Rodzina Kowalskich wybiera się na tygodniowe wakacje (7 nocy). Zarezerwowali pokój na 45 dni przed przyjazdem w cenie 180 EUR za noc. Przyjeżdżają pociągiem, więc parking ich nie interesuje. Mają dwójkę dzieci, z czego jedno to niemowlak. Zostawili aż 3 specjalne życzenia w systemie: łóżeczko dziecięce, pokój z widokiem na basen i dodatkowe ręczniki. Pan Kowalski dzwonił również dopytać, czy w okolicy jest dobry weterynarz, bo pod ich nieobecność w domu choruje im legwan.

**D. "Turysta Budżetowy"**
> Anna, studentka z wymiany z Hiszpanii. Rezerwuje najtańszy pokój za 65 EUR za noc na zaledwie 10 dni przed terminem. Zostaje na 2 noce, żeby zwiedzić miasto z polecenia znajomych. Nie ma auta. Żadnych życzeń specjalnych. Jej ulubiony zespół to Arctic Monkeys i ma nadzieję, że grają gdzieś w pobliżu jakiś sekretny koncert.

**E. "Early Bird"**
> Tomek zarezerwował swój wyjazd na 60 dni do przodu. Cena jest całkiem atrakcyjna: 95 EUR za noc. Wpada do miasta na 3 noce. Przyjeżdża swoim kombi, więc koniecznie zaznaczył opcję parkingu. Jego jedyne specjalne życzenie to wegańskie śniadanie ze względu na nową dietę. Posiada złotą kartę lojalnościową z innej sieci hoteli, o czym nie omieszkał z dumą wspomnieć w mailu. Zawsze płaci z góry kartą kredytową.

**F. "Niepewny"**
> Tajemniczy gość z Francji. Rezerwacja zrobiona bardzo wcześnie (150 dni przed), wziął bardzo drogi pokój z balkonem (210 EUR/noc) na 4 noce. Zero życzeń specjalnych, nie zaznaczył parkingu. Pytał tylko w osobnym mailu o politykę zwrotów w razie odwołania lotu oraz o to, czy w recepcji można kupić znaczki pocztowe, żeby wysłać pocztówkę do babci.

**G. "Stały Klient"**
> Pani Krystyna bywa w tym hotelu regularnie (to jej 12. wizyta). Rezerwuje zawsze na ostatnią chwilę – tym razem na 5 dni przed przyjazdem. Płaci stałą, wynegocjowaną przez firmę stawkę 110 EUR za noc i zatrzymuje się na 2 noce. Nie jeździ samochodem (nie lubi). Zostawiła 2 życzenia w systemie: zawsze prosi o pokój 404 (ten na końcu korytarza) i gazetę poranną pod drzwiami. Uwielbia wieczorami plotkować z barmanem o lokalnej polityce.

---

### Instrukcja (Inference):

Zbuduj w kolejnej komórce z kodem tablicę NumPy (`np.array`) zawierającą dane tych 7 klientów (wymiar $7 \times 5$). Następnie użyj swoich wyuczonych wag $\mathbf{w}$ oraz biasu $b$, aby obliczyć prawdopodobieństwo anulowania.

Kroki w Pythonie:
1. Zbuduj macierz `X_nowi` (uważaj, aby zachować odpowiednią kolejność kolumn!).
2. **Standaryzacja:** `X_nowi_norm = (X_nowi - mean_train) / std_train` (użyj średniej i odchylenia ze zbioru treningowego).
3. **Model:** `z = np.dot(X_nowi_norm, w) + b`
4. **Aktywacja:** `a = 1 / (1 + np.exp(-z))`

**Cel biznesowy:** Wypisz na ekranie prawdopodobieństwa (w procentach) dla każdego z gości A-G. Ustaw próg odcięcia na poziomie 50% ($0.5$). Kto dostanie flagę "WYSOKIE RYZYKO ANULOWANIA", a kto "ZAPEWNE PRZYJEDZIE"?

In [ ]:
# =====================================================================
# ZADANIE 5: PREDYKCJA DLA NOWYCH KLIENTÓW (INFERENCE)
# =====================================================================



## Krok 6: Ewaluacja Modelu – Czy możemy zaufać maszynie?

Gratulacje! Twój model jest już wytrenowany. Widzieliśmy, że całkiem sensownie ocenia ryzyko dla nowych gości. Ale wyobraź sobie, że idziesz z tym algorytmem do Zarządu Hotelu. Dyrektor zadaje Ci jedno, kluczowe pytanie: **"W ilu procentach ten system ma rację?"**

Aby na to odpowiedzieć, musimy sprawdzić, jak nasz wyuczony model radzi sobie na historycznych danych. Zastosujemy standardowy próg odcięcia (jeśli prawdopodobieństwo $> 0.5$, przewidujemy anulację) i zderzymy przewidywania maszyny z brutalną rzeczywistością.

W klasyfikacji binarnej ocena modelu to jednak coś więcej niż tylko podanie ogólnej skuteczności w procentach. Błąd błędowi nierówny! Rozróżniamy 4 konkretne scenariusze, które tworzą tzw. **Macierz Pomyłek (Confusion Matrix)**:



1. **Prawdziwie Pozytywne (True Positives - TP):**
   * **Model:** Twierdzi, że klient anuluje.
   * **Rzeczywistość:** Klient faktycznie anulował.
   * **Biznes:**  Sukces! Hotel może bezpiecznie odsprzedać pokój innej osobie.

2. **Prawdziwie Negatywne (True Negatives - TN):**
   * **Model:** Twierdzi, że klient przyjedzie.
   * **Rzeczywistość:** Klient przyjechał.
   * **Biznes:**  Sukces. Normalny przebieg pracy hotelu.

3. **Fałszywe Alarmy (False Positives - FP) – Błąd I rodzaju:**
   * **Model:** Twierdzi, że klient anuluje.
   * **Rzeczywistość:** Klient jednak przyjechał!
   * **Biznes:**  KATASTROFA. Pokój został odsprzedany (tzw. overbooking), a oryginalny gość stoi w recepcji z walizkami. Gigantyczne koszty wizerunkowe.

4. **Przeoczenia (False Negatives - FN) – Błąd II rodzaju:**
   * **Model:** Twierdzi, że klient przyjedzie.
   * **Rzeczywistość:** Klient anuluje w ostatniej chwili.
   * **Biznes:** Strata. Pokój stoi pusty, tracimy potencjalny zarobek.

---

###  Twoje Zadanie Programistyczne:

Twoim celem jest policzenie, ile razy nasz model wpadł w każdy z tych 4 scenariuszy na całym zbiorze treningowym.

1. Przepuść całą macierz `X_norm` (wszystkie historyczne rezerwacje) przez swój wytrenowany model, aby uzyskać prawdopodobieństwa.
2. Przekształć te ułamkowe prawdopodobieństwa na twarde decyzje (1 lub 0) korzystając z progu **0.5**.
3. Używając masek logicznych w Pythonie (np. `(predykcje == 1) & (y_data == 0)`), zlicz wartości TP, TN, FP oraz FN.
4. Wyświetl wyniki na ekranie.

In [ ]:
# =====================================================================
# ZADANIE 6: OBLICZANIE MACIERZY POMYŁEK
# =====================================================================


## Krok 7: Zarządzanie Przychodami (Revenue Management) – Maksymalizacja Zysku

Do tej pory patrzyliśmy na nasz model przez pryzmat kosztów błędów. Ale Zarząd Hotelu interesuje przede wszystkim jedno: **ile pieniędzy będzie w kasie na koniec dnia?**

Przejdźmy na liczenie całkowitego zysku. Załóżmy, że standardowa cena za pokój to 100 EUR. Zbudujmy naszą **Macierz Zysków (Profit Matrix)** dla każdego z 4 scenariuszy:

1. **Prawdziwie Negatywne (TN) – Zwykły gość:**
   * Gość przyjeżdża, my trzymaliśmy dla niego pokój.
   * Zysk: **+100 EUR**
2. **Prawdziwie Pozytywne (TP) – Udana odsprzedaż:**
   * Model zgadł, że gość anuluje. Zwalniamy pokój w systemie i odsprzedajemy go komuś innemu.
   * Zysk: **+100 EUR**
3. **Przeoczenie (FN) – Pusty pokój:**
   * Model uznał, że gość przyjedzie, ale ten zrezygnował w ostatniej chwili. Pokój stoi pusty.
   * Zysk: **0 EUR**
4. **Fałszywy Alarm (FP) – Overbooking (KATASTROFA):**
   * Model kazał odsprzedać pokój, więc wzięliśmy 100 EUR od nowego klienta. Niestety, pierwotny gość przyjechał! Musimy zapłacić mu 300 EUR odszkodowania (taksówka, inny hotel).
   * Zysk netto: 100 EUR - 300 EUR = **-200 EUR** (Jesteśmy na minusie!)

### Twoje Wyzwanie: Szukamy Finansowego Szczytu

Twój cel to znalezienie takiego progu odcięcia, który wygeneruje największy przychód dla hotelu. W poniższej komórce napisz pętlę testującą progi od $0.00$ do $1.00$.

Dla każdego progu oblicz:
$$\text{Całkowity Zysk} = (TP \cdot 100) + (TN \cdot 100) + (FP \cdot (-200)) + (FN \cdot 0)$$

Znajdź próg, przy którym **kasa hotelu jest najpełniejsza**!

In [ ]:
# =====================================================================
# ZADANIE 7: MAKSYMALIZACJA ZYSKU
# =====================================================================
